In [23]:
from datetime import date
from data_engineering.filters import pia_or_american_ladd_icao
from data_engineering.flights.flight_type import FlightIndex, flights_df_to_flights, get_flights
import polars as pl
start_dt = date(2024, 1, 1)
end_dt = date(2026, 8, 24)
df_flights = get_flights(start_dt, end_dt)
df_flights = df_flights.filter(pia_or_american_ladd_icao())
flights = flights_df_to_flights(df_flights)
flight_index = FlightIndex(flights, None)

In [24]:
import pickle
path = '/Volumes/T2-SSD/planequery/data/tests/test_pia/test_real_flights_2/test_pia_test_real_flights_2_20260827_130429/mappings.pkl'
with open(path, "rb") as f:
    mappings = pickle.load(f)

In [25]:
from pia.check_mappings import check_mappings
failed_icaos = check_mappings(flight_index, mappings)

Flights a03bff_2025-09-24_13-25 and a03bc5_2025-09-24_11-34 overlap in time but are assigned to the same real aircraft a11784
Flights a03bff_2025-09-24_14-45 and a03bc5_2025-09-24_11-34 overlap in time but are assigned to the same real aircraft a11784
Flights a03bc5_2025-09-24_11-34 and a03bff_2025-09-24_13-25 overlap in time but are assigned to the same real aircraft a11784
Flights a03bc5_2025-09-24_11-34 and a03bff_2025-09-24_14-45 overlap in time but are assigned to the same real aircraft a11784
Flights a03ba4_2025-04-14_00-10 and a03c08_2025-05-04_04-10 have a discontinuity in airports: KIAD -> KSJC 
Flights a835af_2025-09-19_06-57 and a03cff_2025-11-12_15-36 have a discontinuity in airports: KAUS -> KIAD 
Flights a03cff_2025-11-12_23-15 and a835af_2025-11-27_07-46 have a discontinuity in airports: KNUQ -> KSJC 
Flights a03cff_2026-01-04_05-32 and a03d63_2026-01-20_19-44 have a discontinuity in airports: KOPF -> KAUS 
Flights a11784_2025-04-11_20-53 and a03bc5_2025-04-17_12-59 have

In [26]:
from pia.check_mappings import real_icao_to_pia_icaos_from_flight_mapping, remove_failed_icaos
real_icao_to_pia_icaos = real_icao_to_pia_icaos_from_flight_mapping(mappings)
remove_failed_icaos(real_icao_to_pia_icaos, failed_icaos)

In [27]:
from pia.check_mappings import stats_on_mappings
stats = stats_on_mappings(flight_index, real_icao_to_pia_icaos)

In [28]:
from pia.ground_truth import get_ground_truth_mappings
df_groundtruth = get_ground_truth_mappings()

In [30]:
stats_df = pl.DataFrame([
    {"icao": icao, **values}
    for icao, values in stats.items()
])
groundtruth_pia_icaos = set(df_groundtruth["pia_icao"].to_list())

stats_df = (
    stats_df.with_columns(
        pl.col("icao")
        .map_elements(
            lambda x: sorted(real_icao_to_pia_icaos.get(x, set())),
            return_dtype=pl.List(pl.Utf8),
        )
        .alias("pia_icaos"),
    )
    .with_columns(
        pl.col("pia_icaos")
        .list.eval(pl.element().filter(pl.element().is_in(groundtruth_pia_icaos)))
        .alias("groundtruth_icaos")
    )
)
with pl.Config(tbl_rows=-1):
    display(stats_df.sort("num_transitions", descending=True))

icao,num_transitions,num_flights,pia_icaos,groundtruth_icaos
str,i64,i64,list[str],list[str]
"""a88cfa""",11,357,"[""a03c1d""]",[]
"""a4d7e5""",7,68,"[""a03957"", ""a0398e"", … ""a10150""]",[]
"""a029e0""",5,202,"[""a03940"", ""a03976"", ""a03eae""]",[]
"""a52321""",4,243,"[""a038dc"", ""a039a0"", … ""a10140""]",[]
"""a88ec3""",4,161,"[""a03925"", ""a03989"", … ""a10125""]",[]
"""a95c57""",4,192,"[""a0393e"", ""a03995"", … ""a1011b""]",[]
"""a816e4""",4,12,"[""a03c0e"", ""a03c10"", … ""a03c3a""]",[]
"""a5864b""",3,259,"[""a03932"", ""a0396a"", ""a1012d""]","[""a1012d""]"
"""ad5288""",3,154,"[""a03906"", ""a03c72"", ""a03c76""]",[]


In [21]:
icao = "a29625"
icaos = [icao] + list(real_icao_to_pia_icaos[icao])

In [31]:
import polars as pl
icaos = ["a0fef5","a765c6"]
COLUMNS = ["icao", "pia", "callsign", "takeoff_time", "landing_time", "takeoff_airport_ident", "landing_airport_ident"]
with pl.Config(tbl_rows=-1):
    display(df_flights.filter(pl.col("icao").is_in(icaos)).sort("takeoff_time").select(COLUMNS))

icao,pia,callsign,takeoff_time,landing_time,takeoff_airport_ident,landing_airport_ident
str,bool,str,datetime[ms],datetime[ms],str,str
"""a0fef5""",true,"""FFL778""",2024-01-12 14:08:46.500,2024-01-12 18:19:07.800,"""NR57""","""KHXD"""
"""a0fef5""",true,"""FFL778""",2024-01-13 13:31:21.640,2024-01-13 14:45:36.040,"""KHXD""","""FL60"""
"""a0fef5""",true,"""FFL778""",2024-01-16 23:16:11.690,2024-01-16 23:53:03.440,"""FL60""","""FL60"""
"""a0fef5""",true,"""FFL778""",2024-01-17 15:45:55.810,2024-01-17 17:10:12.950,"""FL60""","""2J9"""
"""a0fef5""",true,"""FFL778""",2024-01-20 00:49:03.250,2024-01-20 16:19:38.820,"""US-3723""","""FL60"""
"""a0fef5""",true,"""FFL778""",2024-01-28 17:05:35.580,2024-01-28 18:46:02.910,"""FL60""","""49FD"""
"""a0fef5""",true,"""FFL778""",2024-01-31 00:09:54.120,2024-01-31 03:00:26.500,"""US-3723""","""US-3723"""
"""a0fef5""",true,"""FFL778""",2024-01-31 13:57:15.700,2024-01-31 15:41:17.960,"""K54J""","""FL60"""
"""a0fef5""",true,"""FFL778""",2024-02-02 14:05:23.750,2024-02-02 16:07:12.400,"""FL60""","""FL60"""


pia_icao,registration,real_icao
str,str,str
